In [42]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [43]:
from google.cloud import bigquery
import pandas as pd

# Inicializa el cliente de BigQuery
client = bigquery.Client(project='dataton-2024-team-01-cofares')

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas desde BigQuery datos_no_descriptions_eans
# Ejecuta la consulta con LIMIT explícito
query = """
SELECT *
FROM `dataton-2024-team-01-cofares.datos_cofares.temporal_data`
--LIMIT 9000
"""
df = client.query(query).to_dataframe()
print(df.head())

# Leer el archivo df_img_description.parquet para utilizarlo como df
#df = pd.read_parquet("df_img_description.parquet")
#print(df.head())

/Users/gabrielnoguera/Documents/DataHub/app-flask/cofaresapp/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


                                                 uri  \
0  gs://dataton-2024-team-01-cofares-datastore/im...   
1  gs://dataton-2024-team-01-cofares-datastore/im...   
2  gs://dataton-2024-team-01-cofares-datastore/im...   
3  gs://dataton-2024-team-01-cofares-datastore/im...   
4  gs://dataton-2024-team-01-cofares-datastore/im...   

                                         descripcion forma color  \
0  ```json\n{"forma": "cilíndrica", "color": "pla...  None  None   
1  ```json\n{"forma": "cilíndrica", "color": ["bl...  None  None   
2  ```json\n{"forma": "cilíndrica", "color": ["bl...  None  None   
3  ```json\n{"forma": "cilíndrica", "color": ["ma...  None  None   
4  ```json\n{"forma": "rectangular", "color": ["b...  None  None   

  descripcion_visual empaque zona_de_aplicacion  
0               None    None               None  
1               None    None               None  
2               None    None               None  
3               None    None               None  
4   

In [39]:
import json

def clean_and_parse_json(text):
    # Check for None or empty values first
    if text is None or text == '':
        return {
            'forma': None,
            'color': None,
            'descripcion_visual': None,
            'empaque': None,
            'zona_de_aplicacion': None
        }
    
    try:
        # Si es un string, limpiamos y parseamos
        if isinstance(text, str):
            # Eliminar ```json del inicio y ``` del final
            text = text.replace('```json\n', '').replace('\n```', '')
            # Eliminar cualquier espacio en blanco extra
            text = text.strip()
            
            # Si el texto parece estar truncado, intentamos repararlo
            if text.count('{') > text.count('}'):
                text = text + '"}'  # Cerramos el string y el objeto JSON
            
            return json.loads(text)
            
    except json.JSONDecodeError:
        # Si falla, retornamos un diccionario con valores None
        return {
            'forma': None,
            'color': None,
            'descripcion_visual': None,
            'empaque': None,
            'zona_de_aplicacion': None
        }

# Aplicar la función a cada fila
df['descripcion'] = df['descripcion'].apply(clean_and_parse_json)

# Extraer cada campo del diccionario a su respectiva columna
df['forma'] = df['descripcion'].apply(lambda x: x.get('forma'))
df['color'] = df['descripcion'].apply(lambda x: x.get('color'))
df['descripcion_visual'] = df['descripcion'].apply(lambda x: x.get('descripcion_visual'))
df['empaque'] = df['descripcion'].apply(lambda x: x.get('empaque'))
df['zona_de_aplicacion'] = df['descripcion'].apply(lambda x: x.get('zona_de_aplicacion'))

print("\nResultado final:")
print(df.head())

# Verificar registros no nulos
print("\nRegistros no nulos por columna:")
print(df[['forma', 'color', 'descripcion_visual', 'empaque', 'zona_de_aplicacion']].notna().sum())


Resultado final:
                                                 uri  \
0  gs://dataton-2024-team-01-cofares-datastore/im...   
1  gs://dataton-2024-team-01-cofares-datastore/im...   
2  gs://dataton-2024-team-01-cofares-datastore/im...   
3  gs://dataton-2024-team-01-cofares-datastore/im...   
4  gs://dataton-2024-team-01-cofares-datastore/im...   

                                         descripcion        forma  \
0  {'forma': 'cilíndrica', 'color': 'plateado', '...   cilíndrica   
1  {'forma': 'cilíndrica', 'color': ['blanco', 'v...   cilíndrica   
2  {'forma': 'cilíndrica', 'color': ['blanco', 'v...   cilíndrica   
3  {'forma': 'cilíndrica', 'color': ['marrón', 'd...   cilíndrica   
4  {'forma': 'rectangular', 'color': ['blanco', '...  rectangular   

                      color  \
0                  plateado   
1   [blanco, verde, morado]   
2  [blanco, verde, naranja]   
3  [marrón, dorado, blanco]   
4          [blanco, morado]   

                                  descripci

In [40]:
def clean_list_values(value):
    # Manejar valores nulos
    if value is None:
        return None
    
    # Si es una lista
    if isinstance(value, list):
        # Filtrar valores nulos y limpiar
        cleaned = [str(item).lower() for item in value if item and str(item).lower() != 'null']
        # Remover caracteres especiales manteniendo tildes y ñ
        cleaned = [re.sub(r'[^\w\s,áéíóúñ]', '', item).strip() for item in cleaned]
        # Remover strings vacíos
        cleaned = [item for item in cleaned if item]
        return ', '.join(cleaned) if cleaned else None
    
    # Si es un valor simple
    return re.sub(r'[^\w\s,áéíóúñ]', '', str(value).lower()).strip()

# Después de tu código existente, agregar:
import re

# Limpiar cada columna
columns_to_clean = ['forma', 'color', 'descripcion_visual', 'empaque', 'zona_de_aplicacion']
for col in columns_to_clean:
    df[col] = df[col].apply(clean_list_values)

print("\nResultado después de la limpieza:")
print(df[columns_to_clean].head())

# Verificar registros no nulos después de la limpieza
print("\nRegistros no nulos por columna después de la limpieza:")
print(df[columns_to_clean].notna().sum())


Resultado después de la limpieza:
         forma                   color  \
0   cilíndrica                plateado   
1   cilíndrica   blanco, verde, morado   
2   cilíndrica  blanco, verde, naranja   
3   cilíndrica  marrón, dorado, blanco   
4  rectangular          blanco, morado   

                                  descripcion_visual     empaque  \
0  un contenedor de metal con una tapa circular y...  no visible   
1  un frasco blanco con tapa morada, con una etiq...      frasco   
2  botella blanca con etiqueta naranja y verde co...     botella   
3  un frasco de color marrón con una tapa dorada,...      frasco   
4  dispositivo electrónico con pantalla digital y...     ninguno   

  zona_de_aplicacion  
0         no visible  
1               None  
2               None  
3               None  
4            ninguna  

Registros no nulos por columna después de la limpieza:
forma                 8650
color                 8837
descripcion_visual    8837
empaque               8396
z

In [35]:
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"

In [36]:
# Convertir listas a strings para las columnas que lo necesiten
df['forma'] = df['forma'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['color'] = df['color'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['descripcion_visual'] = df['descripcion_visual'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['empaque'] = df['empaque'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['zona_de_aplicacion'] = df['zona_de_aplicacion'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Eliminar la columna 'descripcion' original si aún no lo has hecho
if 'descripcion' in df.columns:
    df = df.drop('descripcion', axis=1)

# Subir el DataFrame a BigQuery
table_id = "datos_cofares.descripciones_completas_temporal"
df.to_gbq(
    destination_table=table_id,
    project_id=project_id,
    if_exists='replace',  # Puede ser 'fail', 'replace' o 'append'
    location='EU'  # O 'US' dependiendo de tu región
)

print(f"Tabla subida exitosamente a {project_id}.{table_id}")

/var/folders/z8/ttpm9zvj1gl8xn76dz5g5qym0000gn/T/ipykernel_13764/1561317481.py:14: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 3584.88it/s]

Tabla subida exitosamente a dataton-2024-team-01-cofares.datos_cofares.descripciones_completas_temporal


In [25]:
# Convertir listas a strings para las columnas que lo necesiten
df['shape'] = df['shape'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['color'] = df['color'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['visual_description'] = df['visual_description'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['packaging'] = df['packaging'].apply(lambda x: str(x) if isinstance(x, list) else x)
df['application_area'] = df['application_area'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Eliminar la columna 'descripcion' original si aún no lo has hecho
if 'descripcion' in df.columns:
    df = df.drop('descripcion', axis=1)

# Subir el DataFrame a BigQuery
table_id = "datos_cofares.descripciones_completas_temporal"
df.to_gbq(
    destination_table=table_id,
    project_id=project_id,
    if_exists='replace',  # Puede ser 'fail', 'replace' o 'append'
    location='EU'  # O 'US' dependiendo de tu región
)

print(f"Tabla subida exitosamente a {project_id}.{table_id}")

/var/folders/z8/ttpm9zvj1gl8xn76dz5g5qym0000gn/T/ipykernel_13764/1904517318.py:14: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 11748.75it/s]

Tabla subida exitosamente a dataton-2024-team-01-cofares.datos_cofares.descripciones_completas_temporal
